# Hospital bed surge forecast

Predicts ward occupancy 24, 48 and 72 hours ahead, so a hospital can see a
surge coming rather than discovering it when the beds run out.

A forecast is only worth having if it beats the obvious guess. The obvious
guess here is "tomorrow looks like today" — naive persistence — and that is
reported alongside every horizon.

In [ ]:
import subprocess, sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
print("project root:", ROOT)

## 1. Data

Generated: two years of daily ward occupancy across five hospitals, with
seasonal dengue pressure and injected surge events.

This is the corpus most in need of replacement. DGHS publishes real bed
occupancy, but monthly and per facility, where the model needs daily and per
ward. That requires a data-sharing agreement with a hospital.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "ml" / "generate_bed_logs.py")], check=True)

In [ ]:
import pandas as pd

beds = pd.read_csv(ROOT / "data" / "surge" / "bed_utilization.csv")
print(f"{len(beds):,} rows  {beds.date.min()} to {beds.date.max()}")
display(beds.groupby("ward_type")[["capacity", "occupied"]].mean().round(1))

## 2. Train

In [ ]:
subprocess.run([sys.executable, str(ROOT / "ml" / "train_surge_model.py")], check=True)

In [ ]:
metrics = json.loads((ROOT / "backend/app/ai/artifacts/surge_metrics.json").read_text())
for horizon in ("h1", "h2", "h3"):
    row = metrics[horizon]
    better = row["naive_persistence_mae"] - row["test_mae_beds"]
    print(f"{horizon}: MAE {row['test_mae_beds']:.2f} beds "
          f"(naive {row['naive_persistence_mae']:.2f}, better by {better:.2f})")